# Lab 01 (solution): Tokenization and embeddings from scratch

Reference implementation. Build byte-pair encoding and count-based embeddings, and see why token counts fill the context window and how cosine similarity tracks meaning. Concept: [tokens-and-embeddings.md](../../../concepts/llm/tokens-and-embeddings.md); math: [math-foundations/01](../../../math-foundations/01-embeddings-and-similarity.md).

## Step 0: Tokenize with BPE

In [ ]:
import sys; sys.path.insert(0, "..")  # modules (bpe.py, embeddings.py) live in the lab root
from bpe import train, encode, decode, _CORPUS
# A model sees token ids, not words. BPE decides what a token is: merge the most frequent adjacent
# pair, repeatedly. Common words become one token; rare words fall back to pieces.
merges = train(_CORPUS, num_merges=10)
print("learned merges:", merges[:6], "...")
for w in ["newer", "lowest", "colder"]:  # last is unseen
    toks = encode(w, merges)
    print(f"  {w:8} -> {toks}  ({len(toks)} tokens)  round-trip={decode(toks)==w}")

## Step 1: The tokenization tax

In [ ]:
# The "tokenization tax": an unusual word costs more tokens, so more context and more money.
common = encode("newer", merges); rare = encode("colder", merges)
print(f"trained 'newer': {len(common)} token, unseen 'colder': {len(rare)} tokens "
      f"-> {len(rare)}x the context for one word")

## Step 2: Embeddings from co-occurrence counts

In [ ]:
from embeddings import Embeddings
# An embedding is a vector where "close in meaning" is "close in space." Co-occurrence counts already
# are one: words in similar contexts get similar vectors.
raw = Embeddings(weighting="raw")
print(f"raw counts:  cos(cat,dog)={raw.similarity('cat','dog'):.2f}  cos(cat,car)={raw.similarity('cat','car'):.2f}")

## Step 3: PPMI sharpens the signal

In [ ]:
# Raw counts are dominated by frequent words like "the". PPMI re-weights by how much more than
# chance two words co-occur, suppressing ubiquitous words and surfacing meaning.
emb = Embeddings(weighting="ppmi")
print(f"PPMI:        cos(cat,dog)={emb.similarity('cat','dog'):.2f}  cos(cat,car)={emb.similarity('cat','car'):.2f}")
print("nearest to 'cat':", emb.nearest("cat", k=3))

## What you built

The two operations every LLM starts with, from scratch. **BPE** turns text into a small, fixed set of token ids by merging frequent character pairs; a trained word becomes one token while an unseen word falls back to smaller pieces, which is why token counts (not word counts) fill the context window and why rare strings cost more. **Embeddings** turn tokens into vectors where cosine similarity tracks meaning; raw co-occurrence already works, and PPMI sharpens it by suppressing frequent words, taking cos(cat,car) from 0.77 to 0.07 while cos(cat,dog) stays high.

**Where this simplifies:** real tokenizers use a byte-level base so nothing is out-of-vocabulary, and real embeddings are dense vectors learned by a network (word2vec, then the encoders behind retrieval) rather than raw count rows - but the geometry is identical, and the cosine you used here is the exact operation a vector database runs at scale. Concept: [concepts/llm/tokens-and-embeddings.md](../../../concepts/llm/tokens-and-embeddings.md); math: [math-foundations/01](../../../math-foundations/01-embeddings-and-similarity.md).